# 🧹 DataCo Supply Chain Cleaning Pipeline: **Silver Layer**
---

### 📌 Architecture Overview
This notebook implements the **Silver Layer Cleaning and Standardization** phase. It reads raw transactional logs from the Bronze Delta table, applies robust cleaning transformations, casts columns to their optimal data types, handles geographic and financial anomalies, and saves the cleaned dataset to the Silver Delta table.

* **Source Table:** `workspace.dataco_bronze.supply_chain_raw` (Delta Lake)
* **Target Table:** `workspace.dataco_silver.supply_chain_cleaned` (Delta Lake)

```
 +------------------------------+      +---------------------------+      +-----------------------------------+
 |  Bronze Layer (Delta Lake)   |      |    Databricks Workspace   |      |     Silver Layer (Delta Lake)     |
 |                              |      |                           |      |                                   |
 |   bronze_supply_chain_raw    | ---> |  Data Cleansing, Casing   | ---> |     supply_chain_cleaned          |
 |   (raw ingested structure)   |      |  Type Casts, Deduplication|      |  (standardized reporting schema)  |
 +------------------------------+      +---------------------------+      +-----------------------------------+
```

---
### ⚙️ Implementation Steps
1. **Drop Constant Columns:** Drops zero-variance columns (`customer_email`, `customer_password`, `product_description`, and `product_status`) to free space and simplify reporting.
2. **Whitespace Trimming & Placeholder Nullification:** Trims spaces and case-insensitively nullifies empty strings or string null representations (like `"null"` or `"none"`).
3. **Geographic Standardization:** Pads customer zip codes to standard US 5-character formats. Pads destination zip codes ONLY for US orders to keep international zip codes intact.
4. **Financial Type Casts:** Converts monetary fields to `DecimalType(10, 2)` to eliminate floating-point calculation errors in Tableau dashboards.
5. **Casing Normalization:** Capitalizes name, product, and city columns cleanly, while setting states and statuses to UPPERCASE.
6. **Simple Deduplication:** Removes duplicate records based on `order_item_id` using a clean, single-line Spark command.
7. **Audit Logging:** Appends a clean `cleaned_at` timestamp before saving (no personal user emails or bucket paths are written, keeping it safe for public GitHub sharing).


## 🛠️ Step 1: Configuration & Parameterization
Define widgets to configure target schemas and catalog locations dynamically.


In [0]:
# Define widgets for Silver notebook configuration
dbutils.widgets.text("source_catalog", "workspace", "Source Catalog (Bronze)")
dbutils.widgets.text("source_schema", "dataco_bronze", "Source Schema (Bronze)")
dbutils.widgets.text("source_table", "supply_chain_raw", "Source Table Name (Bronze)")

dbutils.widgets.text("target_catalog", "workspace", "Target Catalog (Silver)")
dbutils.widgets.text("target_schema", "dataco_silver", "Target Schema (Silver)")
dbutils.widgets.text("target_table", "supply_chain_cleaned", "Target Table Name (Silver)")
dbutils.widgets.dropdown("write_mode", "overwrite", ["overwrite", "append"], "Write Mode")


## 📖 Step 2: Read Bronze Table
Load target parameters from the widgets and query the Bronze Delta table.


In [0]:
# Retrieve parameter values from widgets
source_catalog = dbutils.widgets.get("source_catalog").strip()
source_schema = dbutils.widgets.get("source_schema").strip()
source_table = dbutils.widgets.get("source_table").strip()

target_catalog = dbutils.widgets.get("target_catalog").strip()
target_schema = dbutils.widgets.get("target_schema").strip()
target_table = dbutils.widgets.get("target_table").strip()
write_mode = dbutils.widgets.get("write_mode").strip()

# Build full source path and load the Bronze data
if source_catalog.lower() != "hive_metastore":
    source_path = f"`{source_catalog}`.`{source_schema}`.`{source_table}`"
else:
    source_path = f"`{source_schema}`.`{source_table}`"

print(f"📖 Reading Bronze Delta table: {source_path}")
df_bronze = spark.read.table(source_path)
print(f"✅ Loaded {df_bronze.count():,} records from Bronze.")


## 🗑️ Step 3: Column Name Sanitization & Drop Junk Columns
Standardizes columns to `snake_case` if they were ingested with raw formats in the Bronze layer. Then, drops empty and constant columns that carry no analytical value.


In [0]:
import re

# 1. Define Column Name Clean Function for Resiliency
def clean_column_name(col_name: str) -> str:
    clean = col_name.strip().lower()
    clean = clean.replace(" (real)", "_real")
    clean = clean.replace(" (scheduled)", "_scheduled")
    clean = clean.replace(" (dateorders)", "")
    clean = re.sub(r'[^a-zA-Z0-9_]', '_', clean)
    clean = re.sub(r'_+', '_', clean)
    return clean.strip('_')

# Apply clean function to columns to ensure snake_case layout
sanitized_cols = [clean_column_name(c) for c in df_bronze.columns]
df_clean = df_bronze.toDF(*sanitized_cols)
print("✅ Standardized all columns to snake_case schema.")

# 2. Drop constant/empty columns
junk_cols = [
    "customer_email",      # Masked to XXXXXXXXX
    "customer_password",   # Masked to XXXXXXXXX
    "product_description",  # 100% Null
    "product_status"        # All values are 0 (active)
]

print(f"🗑️ Dropping junk/empty columns: {junk_cols}")
df_clean = df_clean.drop(*junk_cols)
print(f"✅ Schema reduced to {len(df_clean.columns)} columns.")


## 🧽 Step 4: String Column Trimming & Placeholder Nullification
Trims leading/trailing spaces from all string columns, and converts empty text values (`""`) or text placeholders (`"null"`, `"none"`, etc. regardless of capitalization) into true SQL `NULL`s.


In [0]:
from pyspark.sql.functions import col, trim, when, lit, lower

# Identify all string type columns
string_cols = [f.name for f in df_clean.schema.fields if f.dataType.simpleString() == "string"]

# Standardize empty spaces and text-literal null markers to true SQL null values (case-insensitively)
for c in string_cols:
    df_clean = df_clean.withColumn(
        c,
        when(
            (trim(col(c)) == "") | 
            (lower(trim(col(c))).isin("null", "none", "nan", "undefined", "?")), 
            lit(None)
        ).otherwise(trim(col(c)))
    )

print("✅ String column trimming and placeholder nullification complete.")


## ✏️ Step 5: Text Casing Normalization & Zipcode Padding
Standardizes capitalization to eliminate duplicate categorical entities in Tableau dashboards.
* **Casing:** Title Case (`initcap`) for general entities and names. UPPERCASE (`upper`) for state and order statuses.
* **Zipcodes:** Pads customer zip codes (entirely in US/PR) to standard 5 digits. Pads order zip codes ONLY for US orders (`Estados Unidos`) to prevent corrupting foreign postal code shapes.


In [0]:
from pyspark.sql.functions import col, lit, when, initcap, upper, lpad, coalesce

# 1. Define columns for Title Case Casing
title_case_cols = [
    "customer_fname", "customer_lname", "customer_city", "customer_street",
    "order_city", "order_state", "order_country", "category_name", 
    "department_name", "product_name", "shipping_mode"
]

for c in title_case_cols:
    df_clean = df_clean.withColumn(c, initcap(col(c)))

# 2. Standardize State, Status, and Type Columns to UPPERCASE
upper_case_cols = ["customer_state", "order_status", "type"]
for c in upper_case_cols:
    df_clean = df_clean.withColumn(c, upper(col(c)))

# 3. Standardize Customer Country representations
df_clean = df_clean.withColumn(
    "customer_country",
    when(upper(col("customer_country")).isin("EE. UU.", "EE.UU.", "EE.UU"), lit("EE. UU."))
    .otherwise(initcap(col("customer_country")))
)

# 4. Impute missing customer last names with 'Unknown' for BI report cleanliness
df_clean = df_clean.withColumn("customer_lname", coalesce(col("customer_lname"), lit("Unknown")))

# 5. Standardize Zipcodes (zero-padding US and Puerto Rico areas)
df_clean = df_clean.withColumn("customer_zipcode", lpad(col("customer_zipcode"), 5, "0"))

# Order destinations are international. Pad zipcode ONLY for US orders ('Estados Unidos')
df_clean = df_clean.withColumn(
    "order_zipcode",
    when(col("order_country") == "Estados Unidos", lpad(col("order_zipcode"), 5, "0"))
    .otherwise(col("order_zipcode"))
)

print("✅ Casing normalization, default name imputation, and US zipcode padding applied.")


## 📊 Step 6: DataType Enforcements & Date Logical Validations
Casts monetary variables to precise `DecimalType(10, 2)` to prevent floating-point calculation errors in Tableau aggregates. Converts date strings into Spark timestamps and runs a logical validation check.


In [0]:
from pyspark.sql.types import DecimalType, TimestampType
from pyspark.sql.functions import col

# 1. Cast monetary/sales columns to precise DecimalType(10, 2)
monetary_cols = [
    "sales", "benefit_per_order", "sales_per_customer", "order_item_total", 
    "order_profit_per_order", "order_item_product_price", "product_price", 
    "order_item_discount"
]

for c in monetary_cols:
    df_clean = df_clean.withColumn(c, col(c).cast(DecimalType(10, 2)))

# 2. Explicitly cast date fields to TimestampType (if not already handled in Bronze)
df_clean = df_clean \
    .withColumn("order_date", col("order_date").cast(TimestampType())) \
    .withColumn("shipping_date", col("shipping_date").cast(TimestampType()))

# 3. Validate temporal chronology (Shipping date must be >= Order date)
anomalous_dates_count = df_clean.filter(col("shipping_date") < col("order_date")).count()
print(f"⏳ Date chronology check: Found {anomalous_dates_count} anomalous rows where shipping_date < order_date.")
if anomalous_dates_count > 0:
    print("⚠️ WARNING: Shipping date anomalies detected. Shipping date must not occur before order date.")
else:
    print("✅ Date chronology validation check passed successfully.")


## 🔑 Step 7: Deduplicate Data on Granular Transaction Key
Each row in the dataset represents a single transaction line item, identified by the unique column `order_item_id`. We run a standard, highly readable single-line PySpark command to drop any duplicate rows based on this key.


In [0]:
from pyspark.sql.functions import col

# Deduplicate on order_item_id
total_rows_pre = df_clean.count()
df_final = df_clean.dropDuplicates(["order_item_id"])
total_rows_post = df_final.count()

duplicates_dropped = total_rows_pre - total_rows_post
print(f"🔑 Deduplication: Dropped {duplicates_dropped:,} duplicate records based on order_item_id.")
print(f"✅ Retained {total_rows_post:,} unique transaction rows.")


## 🛡️ Step 8: Clean Lineage Timestamp Enrichment
Appends a processing timestamp to log when this file was cleaned. We avoid logging private emails or local folder paths to keep the repository safe for public GitHub sharing.


In [0]:
from pyspark.sql.functions import current_timestamp

# Add Silver cleaning timestamp
df_silver = df_final.withColumn("cleaned_at", current_timestamp())
print("🛡️ Added Silver lineage metadata column: cleaned_at.")


## 💾 Step 9: Save Cleaned Data to Silver Delta Table
Creates the Silver target database schema if it does not exist, and writes the standardized DataFrame to the target Silver Delta table.


In [0]:
# Resolve target catalog and schema database paths
if target_catalog and target_catalog.lower() != "hive_metastore":
    full_target_path = f"`{target_catalog}`.`{target_schema}`.`{target_table}`"
    spark.sql(f"CREATE DATABASE IF NOT EXISTS `{target_catalog}`.`{target_schema}`")
else:
    full_target_path = f"`{target_schema}`.`{target_table}`"
    spark.sql(f"CREATE DATABASE IF NOT EXISTS `{target_schema}`")

print(f"💾 Saving clean dataset to Silver Delta Table: {full_target_path}")

# Write cleaned dataframe to target
(df_silver.write
    .format("delta")
    .mode(write_mode)
    .option("mergeSchema", "true")
    .saveAsTable(full_target_path)
)

print(f"🚀 Silver data load complete in {write_mode.upper()} mode.")


## ⚡ Step 10: Delta Lake Storage Optimization
Performs compaction on small storage files and Z-Orders on `order_date` to accelerate Tableau query speeds.


In [0]:
# Run compaction and Z-Order index optimization
optimize_query = f"OPTIMIZE {full_target_path} ZORDER BY (order_date)"
print(f"⚡ Executing Compaction: {optimize_query}")
spark.sql(optimize_query)
print("⚡ Delta Lake storage optimization complete.")


## 🔎 Step 11: Data Load Validation & Quality Checks
Performs a quick validation check on the clean Silver table schema and previews a sample of records.


In [0]:
from pyspark.sql.functions import length, col

# 1. Load target to run diagnostics
df_target = spark.read.table(full_target_path)
total_target_records = df_target.count()
print(f"📊 Total records saved in Silver: {total_target_records:,}")

# 2. Check for remaining trailing spaces in product_name
whitespace_records = df_target.filter(col("product_name").endswith(" ")).count()
print(f"🔎 Trailing space check (product_name): Found {whitespace_records} records with trailing spaces.")
if whitespace_records > 0:
    print("⚠️ Quality failure: Trailing spaces remain in key text columns.")
else:
    print("✅ Quality check passed: Trailing spaces resolved in product_name.")

# 3. Verify dropped columns are not present
existing_cols = df_target.columns
dropped_present = [c for c in junk_cols if c in existing_cols]
print(f"🔎 Column drop check: Found dropped columns {dropped_present} in target schema.")
if len(dropped_present) > 0:
    print("⚠️ Quality failure: Constant/junk columns were not successfully removed.")
else:
    print("✅ Quality check passed: Constant/junk columns dropped successfully.")

# 4. Display sample of cleaned data
print("👀 Preview of Cleaned Silver Dataset:")
display(df_target.limit(5))
